# 🧬 Machine Learning on QSAR Data
### 20+ Models: XGBoost, LightGBM, CatBoost, TabNet, Neural Nets & More!

**Companion notebook for the Omixium YouTube tutorial**  
📺 [Watch the video](https://www.youtube.com/watch?v=BC8r-xb9BpA)  
📂 [GitHub repository](https://github.com/YOUR_USERNAME/QSAR-ML-Drug-Discovery)

---

## What This Notebook Covers

1. **Data Collection** — Fetch bioactivity data from ChEMBL for Acetylcholinesterase (AChE)
2. **Preprocessing** — Clean, deduplicate, and convert IC50 → pIC50
3. **Exploratory Data Analysis** — Distribution plots, chemical space visualization
4. **Featurization** — Morgan fingerprints + RDKit descriptors
5. **Train/Test Split** — 80/20 split with reproducible seeding
6. **Model Training** — 20+ regression models, benchmarked side-by-side
7. **Hyperparameter Tuning** — Optuna optimization on the top 3 models
8. **Explainability** — SHAP feature importance for the best model
9. **Model Persistence** — Save the winning model for future predictions

> 💡 **Tip**: Run this notebook top-to-bottom in Google Colab or locally with the `qsar-ml` conda environment (see repository README for setup instructions).


## 1️⃣ Setup & Installation

In [ ]:
# Run this cell first if working in Google Colab
# (Skip if you've already set up the local conda environment from environment.yml)

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q rdkit chembl-webresource-client
    !pip install -q xgboost lightgbm catboost
    !pip install -q pytorch-tabnet optuna shap
    print("✅ Colab dependencies installed")
else:
    print("ℹ️ Running locally — make sure you've activated the 'qsar-ml' conda environment")


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import time
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# RDKit
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, Draw
from rdkit.Chem.Draw import IPythonConsole

# sklearn
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Reproducibility
SEED = 42
np.random.seed(SEED)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

print("✅ All core libraries imported successfully")
print(f"RDKit version check passed")


In [ ]:
# Add the repository's src/ directory to the path so we can import our modules
sys.path.append(os.path.abspath(".."))

from src.data.fetch_chembl import fetch_bioactivity_data, list_common_targets
from src.data.preprocess import preprocess_bioactivity, ic50_to_pic50, activity_class
from src.features.fingerprints import compute_morgan_fingerprints, compute_maccs_keys
from src.features.descriptors import compute_rdkit_descriptors, compute_lipinski_descriptors, LIPINSKI_DESCRIPTORS
from src.models.train import get_all_models, run_benchmark, compute_metrics
from src.models.evaluate import (
    evaluate_model, plot_actual_vs_predicted, plot_benchmark_comparison,
    plot_shap_summary
)
from src.visualization.eda_plots import (
    plot_pic50_distribution, plot_feature_correlation_heatmap, plot_chemical_space
)

print("✅ Project modules imported successfully")


## 2️⃣ Data Collection from ChEMBL

We'll use **Acetylcholinesterase (AChE)** — ChEMBL Target ID `CHEMBL220` — as our example target.
AChE inhibitors are a key drug class for Alzheimer's disease, and this dataset is large and well-curated,
making it a popular QSAR benchmark.

You can swap in any other target by changing `TARGET_CHEMBL_ID` below.


In [ ]:
# Preview some commonly used QSAR benchmark targets
list_common_targets()


In [ ]:
TARGET_CHEMBL_ID = "CHEMBL220"   # Acetylcholinesterase
BIOACTIVITY_TYPE = "IC50"

RAW_DATA_PATH = "../data/raw/acetylcholinesterase_raw.csv"

# Fetch from ChEMBL (this queries the live API — takes 1-2 minutes)
df_raw = fetch_bioactivity_data(
    target_chembl_id=TARGET_CHEMBL_ID,
    bioactivity_type=BIOACTIVITY_TYPE,
    output_path=RAW_DATA_PATH,
)

print(f"Fetched {len(df_raw)} raw bioactivity records")
df_raw.head()


## 3️⃣ Data Preprocessing

Raw ChEMBL data needs substantial cleaning before it's ready for modeling:

1. Remove rows with missing SMILES or IC50 values
2. Keep only exact measurements (`standard_relation == "="`), dropping `>`/`<` censored data
3. Standardize all units to nanomolar (nM)
4. Deduplicate compounds (median IC50 if a compound has multiple measurements)
5. Convert **IC50 (nM) → pIC50** via: `pIC50 = -log10(IC50_M)`
6. Filter to a sensible pIC50 range (removes potential outliers/errors)

### Why pIC50 instead of raw IC50?

IC50 values span many orders of magnitude (e.g., 0.1 nM to 100,000 nM), creating a highly skewed
target distribution that's hard for most regressors to model well. The negative log transform
(pIC50) compresses this range into a roughly normal distribution centered around 4–9, which is
far more tractable for regression.


In [ ]:
CLEAN_DATA_PATH = "../data/processed/clean_dataset.csv"

df_clean = preprocess_bioactivity(
    df_raw,
    pic50_min=3.0,
    pic50_max=12.0,
    output_path=CLEAN_DATA_PATH,
    verbose=True,
)

df_clean.head()


In [ ]:
# Add activity class labels (useful for later classification-style analysis)
df_clean = activity_class(df_clean, active_threshold=6.0, inactive_threshold=5.0)
df_clean["activity_class"].value_counts()


## 4️⃣ Exploratory Data Analysis

In [ ]:
plot_pic50_distribution(
    df_clean["pIC50"].values,
    active_threshold=6.0,
    inactive_threshold=5.0,
    save_path="../results/figures/pic50_distribution.png",
)


In [ ]:
# Compute Lipinski descriptors for EDA (not yet our ML feature matrix)
X_lip, lip_names = compute_lipinski_descriptors(df_clean["canonical_smiles"].tolist())
df_lipinski = pd.DataFrame(X_lip, columns=lip_names)
df_lipinski["pIC50"] = df_clean["pIC50"].values

df_lipinski.describe().T


In [ ]:
plot_feature_correlation_heatmap(
    df_lipinski,
    feature_cols=["MolWt", "MolLogP", "NumHDonors", "NumHAcceptors",
                  "NumRotatableBonds", "TPSA", "pIC50"],
    save_path="../results/figures/lipinski_correlation.png",
)


In [ ]:
# Visualize chemical structures of the 5 most potent compounds
top5 = df_clean.nlargest(5, "pIC50")
mols = [Chem.MolFromSmiles(s) for s in top5["canonical_smiles"]]
legends = [f"pIC50={v:.2f}" for v in top5["pIC50"]]

Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(220, 220), legends=legends)


## 5️⃣ Molecular Featurization

We convert SMILES strings into numeric feature vectors that ML models can consume. This notebook
uses two complementary representations:

- **Morgan (ECFP4) fingerprints** — 2048-bit circular fingerprints capturing local substructure patterns
- **RDKit 2D descriptors** — ~200 interpretable physicochemical properties (MW, LogP, TPSA, etc.)

We'll train primarily on Morgan fingerprints (the standard QSAR baseline) and use descriptors for
the explainability section, since interpretable feature names make SHAP plots more informative.


In [ ]:
smiles_list = df_clean["canonical_smiles"].tolist()
y = df_clean["pIC50"].values

print(f"Featurizing {len(smiles_list)} compounds...")

# Morgan fingerprints (primary feature set for modeling)
X_morgan = compute_morgan_fingerprints(smiles_list, radius=2, n_bits=2048)
print(f"Morgan fingerprints shape: {X_morgan.shape}")

# RDKit descriptors (secondary feature set for explainability)
X_desc, desc_names = compute_rdkit_descriptors(smiles_list)
print(f"RDKit descriptors shape: {X_desc.shape}")


In [ ]:
# Save processed feature matrices
morgan_cols = [f"Morgan_{i}" for i in range(X_morgan.shape[1])]
df_morgan = pd.DataFrame(X_morgan, columns=morgan_cols)
df_morgan.insert(0, "canonical_smiles", smiles_list)
df_morgan["pIC50"] = y
df_morgan.to_csv("../data/processed/morgan_fps.csv", index=False)

df_descriptors = pd.DataFrame(X_desc, columns=desc_names)
df_descriptors.insert(0, "canonical_smiles", smiles_list)
df_descriptors["pIC50"] = y
df_descriptors.to_csv("../data/processed/descriptors.csv", index=False)

print("✅ Saved feature matrices to data/processed/")


## 6️⃣ Train / Test Split

We hold out 20% of compounds for final evaluation, never touched during training or
cross-validation, to get an unbiased estimate of generalization performance.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_morgan, y, test_size=0.2, random_state=SEED
)

print(f"Training set: {X_train.shape[0]} compounds")
print(f"Test set:     {X_test.shape[0]} compounds")


## 7️⃣ Model Training — 20+ Models Benchmarked

This is the core of the tutorial. We train and 5-fold cross-validate over 20+ regression models,
spanning:

- **Linear models**: Linear, Ridge, Lasso, ElasticNet, Bayesian Ridge, Huber
- **Kernel/instance methods**: SVR (RBF & Linear), KNN
- **Trees**: Decision Tree
- **Bagging ensembles**: Random Forest, Extra Trees, Bagging
- **Boosting**: Gradient Boosting, AdaBoost, HistGradientBoosting, **XGBoost**, **LightGBM**, **CatBoost**
- **Neural networks**: MLP (sklearn)
- **Meta-ensembles**: Voting, Stacking

All models are evaluated with identical train/test splits and metrics for a fair comparison.


In [ ]:
results_df = run_benchmark(
    X_morgan, y,
    test_size=0.2,
    cv_folds=5,
    random_state=SEED,
    scale_features=True,
    output_path="../results/reports/benchmark_results.csv",
)

results_df


In [ ]:
plot_benchmark_comparison(
    results_df,
    metric="R2_test",
    save_path="../results/figures/benchmark_r2_comparison.png",
)


## 8️⃣ Deep Learning Models: TabNet & PyTorch MLP

Beyond classical ML, we also train two deep learning architectures specifically suited to
tabular fingerprint data:

- **TabNet** — uses sequential attention to select relevant features at each decision step,
  offering built-in interpretability
- **PyTorch MLP** — a custom 4-layer fully-connected network with BatchNorm + Dropout


In [ ]:
from src.models.tabnet_model import train_tabnet, get_tabnet_feature_importances

# Scale features for TabNet (helps convergence)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Hold out a validation split from training data for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.15, random_state=SEED
)

tabnet_model, tabnet_history = train_tabnet(
    X_tr, y_tr, X_val, y_val,
    n_d=64, n_a=64, n_steps=5,
    max_epochs=150, patience=20,
)


In [ ]:
tabnet_preds = tabnet_model.predict(X_test_scaled).flatten()
tabnet_metrics = compute_metrics(y_test, tabnet_preds)
print("TabNet Test Metrics:", tabnet_metrics)

plot_actual_vs_predicted(
    y_test, tabnet_preds, model_name="TabNet",
    save_path="../results/figures/tabnet_actual_vs_predicted.png",
)


In [ ]:
from src.models.neural_nets import PyTorchQSARRegressor

pytorch_mlp = PyTorchQSARRegressor(
    model_type="mlp",
    hidden_dims=[1024, 512, 256, 128],
    dropout=0.3,
    lr=1e-3,
    max_epochs=200,
    patience=20,
    device="cuda",  # falls back to CPU automatically if unavailable
)
pytorch_mlp.fit(X_train_scaled, y_train)


In [ ]:
mlp_preds = pytorch_mlp.predict(X_test_scaled)
mlp_metrics = compute_metrics(y_test, mlp_preds)
print("PyTorch MLP Test Metrics:", mlp_metrics)

plot_actual_vs_predicted(
    y_test, mlp_preds, model_name="PyTorch MLP",
    save_path="../results/figures/pytorch_mlp_actual_vs_predicted.png",
)


## 9️⃣ Hyperparameter Tuning with Optuna

For the top-performing models from our benchmark (typically the gradient boosting methods),
we use **Optuna** for Bayesian hyperparameter optimization — far more efficient than grid search.


In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
import xgboost as xgb

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": 0,
    }
    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring="r2", n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction="maximize", study_name="xgboost_qsar")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best R²:", study.best_value)
print("Best params:", study.best_params)


In [ ]:
# Train final tuned XGBoost model with the best hyperparameters found
best_xgb = xgb.XGBRegressor(**study.best_params, random_state=SEED, n_jobs=-1, verbosity=0)
best_xgb.fit(X_train, y_train)

xgb_metrics = evaluate_model(best_xgb, X_test, y_test, model_name="Tuned XGBoost")

plot_actual_vs_predicted(
    y_test, xgb_metrics["y_pred"], model_name="Tuned XGBoost",
    save_path="../results/figures/tuned_xgboost_actual_vs_predicted.png",
)


## 🔍 Explainability with SHAP

Now that we have a strong predictive model, we want to understand **why** it makes the
predictions it does. SHAP (SHapley Additive exPlanations) decomposes each prediction into
additive contributions from each input feature.

We re-train on the interpretable RDKit descriptor set for this section, since "Morgan_1453"
is meaningless to a chemist while "MolLogP" or "TPSA" is immediately interpretable.


In [ ]:
X_desc_train, X_desc_test, y_desc_train, y_desc_test = train_test_split(
    X_desc, y, test_size=0.2, random_state=SEED
)

xgb_desc_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbosity=0,
)
xgb_desc_model.fit(X_desc_train, y_desc_train)

desc_metrics = evaluate_model(xgb_desc_model, X_desc_test, y_desc_test,
                              model_name="XGBoost (descriptors)")


In [ ]:
plot_shap_summary(
    xgb_desc_model, X_desc_test,
    feature_names=desc_names,
    model_type="tree",
    max_display=20,
    save_path="../results/figures/shap_summary.png",
)


## 💾 Model Persistence

Finally, we save our best-performing model(s) to disk so they can be reused for inference
on new compounds without retraining.


In [ ]:
import joblib

# Identify the best model from the benchmark table
best_model_name = results_df.iloc[0]["Model"]
print(f"Best model from benchmark: {best_model_name}")

# Re-train it on the full Morgan fingerprint feature set, then persist
all_models = get_all_models()
best_model = all_models[best_model_name]
best_model.fit(X_train, y_train)

joblib.dump(best_model, f"../results/models/{best_model_name}_qsar_model.pkl")
joblib.dump(scaler, "../results/models/feature_scaler.pkl")

print(f"✅ Saved {best_model_name} model to results/models/")


In [ ]:
# Example: load the model back and make a prediction on a new compound
loaded_model = joblib.load(f"../results/models/{best_model_name}_qsar_model.pkl")

new_smiles = ["CC(=O)Oc1ccccc1C(=O)O"]  # Aspirin (just as a syntax example)
new_fp = compute_morgan_fingerprints(new_smiles, radius=2, n_bits=2048)
predicted_pic50 = loaded_model.predict(new_fp)

print(f"Predicted pIC50 for example compound: {predicted_pic50[0]:.3f}")


## 📝 Summary

In this notebook, we built a complete QSAR modeling pipeline:

| Step | What We Did |
|------|-------------|
| Data Collection | Fetched ~thousands of IC50 records from ChEMBL for AChE |
| Preprocessing | Cleaned, deduplicated, converted IC50 → pIC50 |
| Featurization | Morgan fingerprints (2048-bit) + RDKit descriptors (~200) |
| Benchmarking | Trained & cross-validated 20+ ML models |
| Deep Learning | Added TabNet and a custom PyTorch MLP |
| Tuning | Optuna Bayesian optimization on XGBoost |
| Explainability | SHAP summary plots on interpretable descriptors |
| Persistence | Saved the best model with joblib for reuse |

### Next Steps

- Try a different ChEMBL target (see `list_common_targets()` above)
- Add scaffold-based train/test splitting to test generalization to novel chemotypes
- Add a Graph Neural Network (GNN) baseline using PyTorch Geometric
- Deploy the best model behind a simple Streamlit or Gradio app for interactive screening

---
**📺 Full walkthrough**: [YouTube — Omixium](https://www.youtube.com/watch?v=BC8r-xb9BpA)  
**⭐ Star the repo** if you found this useful!
